# 02 ACF, PACF, and Tentative Model Identification

By the end of this notebook, you should be able to:

- compute and interpret ACF and PACF values;
- distinguish cutting off from dying down;
- use ACF/PACF patterns to propose MA(q), AR(p), or mixed ARMA(p, q) working-series models;
- connect the lecture identification table to Python output.

In [1]:
from lite_setup import ensure_packages
await ensure_packages()

Running outside JupyterLite; assuming packages are already installed.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from checks import check, check_between, check_close, check_columns
from boxjenkins_utils import (
    first_difference, second_difference, seasonal_difference, regular_then_seasonal_difference,
    acf_pacf_table, mean_zero_t_test, fit_arima, fit_sarima, parameter_table,
    forecast_table, ljung_box_table, arima_grid_search, plot_series,
    plot_acf_pacf_pair, plot_forecast,
)
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima_process import ArmaProcess
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.precision', 4)
DATA_DIR = Path('data')
towel = pd.read_csv(DATA_DIR / 'paper_towel_sales.csv')
y = towel['Towel_Sales']
z = first_difference(y)

## Sample Autocorrelation Function

For a working series $z_b,z_{b+1},\ldots,z_n$, the sample autocorrelation at lag $k$ measures linear association between observations $k$ periods apart:

$$r_k = \frac{\sum_{t=b}^{n-k}(z_t-\bar z)(z_{t+k}-\bar z)}{\sum_{t=b}^{n}(z_t-\bar z)^2}.$$

A common rough rule is to treat values outside $\pm 2/\sqrt{n}$ as visible spikes. This is a screening rule, not a final proof.

## Manual ACF Calculation and t Values

The lecture does not treat an ACF bar as just a picture. For lag $k$, the sample ACF $r_k$ can be paired with an approximate standard error and t statistic.

For a working series of length $N$, the Bartlett-style standard error used in the lecture/software examples is

$$s_{r_k}=\frac{\sqrt{1+2\sum_{j=1}^{k-1}r_j^2}}{\sqrt{N}}, \quad k\ge 1.$$

Then

$$t_k = \frac{r_k}{s_{r_k}}.$$

For PACF, the common approximate standard error is

$$s_{r_{kk}} = \frac{1}{\sqrt{N}}.$$

A rough screening rule is $|t|>2$, which corresponds visually to being outside about two standard errors. This is a model-identification guide, not a final proof.

In [3]:
from statsmodels.tsa.stattools import acf

# Reproduce the lecture/software example: manual r_3 for the paper towel series.
k = 3
ybar = y.mean()
left = y.iloc[:-k].to_numpy()
right = y.iloc[k:].to_numpy()
numer = np.sum((left - ybar) * (right - ybar))
denom = np.sum((y - ybar) ** 2)
r3_manual = numer / denom
acf_values = acf(y, nlags=10, fft=False)
se_r3 = np.sqrt(1 + 2 * np.sum(acf_values[1:k] ** 2)) / np.sqrt(len(y))
t_r3 = r3_manual / se_r3

pd.Series({
    'manual_r3': r3_manual,
    'statsmodels_r3': acf_values[k],
    'se_r3': se_r3,
    't_r3': t_r3,
}).round(4)

manual_r3         0.8532
statsmodels_r3    0.8532
se_r3             0.1937
t_r3              4.4061
dtype: float64

In [4]:
acf_table_y = acf_pacf_table(y, nlags=14)
acf_table_z = acf_pacf_table(z, nlags=14)
acf_table_y.head(8).round(4)

,lag,ACF,ACF_se,ACF_t,PACF,PACF_se,PACF_t,approx_significant_ACF,approx_significant_PACF
0,0,1.0000,NaN,NaN,1.0000,NaN,NaN,False,False
1,1,0.9626,0.0913,10.5447,0.9626,0.0913,10.5447,True,True
2,2,0.9074,0.1542,5.8849,-0.2611,0.0913,-2.8599,True,True
3,3,0.8532,0.1937,4.4061,0.0439,0.0913,0.4809,True,False
4,4,0.8007,0.2228,3.5941,-0.0263,0.0913,-0.2885,True,False
5,5,0.7428,0.2456,3.0242,-0.1148,0.0913,-1.2574,True,False
6,6,0.6845,0.2637,2.5961,0.0046,0.0913,0.0509,True,False
7,7,0.6277,0.2781,2.2575,-0.0267,0.0913,-0.2927,True,False


In [5]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
plot_acf(y, lags=14, ax=axes[0, 0], title='ACF: original towel sales')
plot_pacf(y, lags=14, ax=axes[0, 1], title='PACF: original towel sales', method='ywmle')
plot_acf(z, lags=14, ax=axes[1, 0], title='ACF: first differences')
plot_pacf(z, lags=14, ax=axes[1, 1], title='PACF: first differences', method='ywmle')
plt.tight_layout()

## PACF: Removing Intervening Lags

The partial autocorrelation at lag $k$ measures the association between $z_t$ and $z_{t-k}$ after accounting for intervening lags $1,\ldots,k-1$. Operationally, ACF shows total lagged correlation, while PACF asks whether lag $k$ still matters after shorter lags are already in the model.

In lecture notation, the sample ACF estimates the theoretical autocorrelation function (TAC), and the sample PACF estimates the theoretical partial autocorrelation function (TPAC).

## Identification Table

Use this table only after the working series is approximately stationary.

| Working-series model | ACF behavior | PACF behavior |
|---|---|---|
| MA(q) | cuts off after lag q | dies down |
| AR(p) | dies down | cuts off after lag p |
| ARMA(p, q) | dies down | dies down |

If both ACF and PACF appear to cut off, decide which one cuts off more clearly and check a small set of candidate models by diagnostics and parsimony.

In [6]:
rng = np.random.default_rng(4031)
def simulate_arma(ar_params=None, ma_params=None, n=400):
    ar_params = [] if ar_params is None else list(ar_params)
    ma_params = [] if ma_params is None else list(ma_params)
    ar = np.r_[1, -np.array(ar_params)]
    ma = np.r_[1, np.array(ma_params)]
    return ArmaProcess(ar, ma).generate_sample(nsample=n, distrvs=rng.standard_normal)
examples = {
    'MA(1), theta=0.65': simulate_arma(ma_params=[0.65]),
    'AR(1), phi=0.65': simulate_arma(ar_params=[0.65]),
    'ARMA(1,1)': simulate_arma(ar_params=[0.55], ma_params=[0.45]),
}
fig, axes = plt.subplots(3, 3, figsize=(12, 9))
for row, (name, series) in enumerate(examples.items()):
    axes[row, 0].plot(series[:120])
    axes[row, 0].set_title(name)
    plot_acf(series, lags=24, ax=axes[row, 1], title='ACF')
    plot_pacf(series, lags=24, ax=axes[row, 2], title='PACF', method='ywmle')
plt.tight_layout()

## Towel First Difference Identification

The original towel series has an ACF that dies down very slowly, so it is not a good stationary working series. The first difference has a strong lag-1 ACF and much shorter memory afterward. The lecture identifies an MA(1) model for the first-differenced working series:

$$z_t = a_t - \theta_1 a_{t-1}, \quad z_t = y_t-y_{t-1}.$$

In ARIMA notation for the original series, that is ARIMA(0, 1, 1).

In [7]:
acf_table_z.head(8).round(4)

,lag,ACF,ACF_se,ACF_t,PACF,PACF_se,PACF_t,approx_significant_ACF,approx_significant_PACF
0,0,1.0000,NaN,NaN,1.0000,NaN,NaN,False,False
1,1,0.3067,0.0917,3.3452,0.3067,0.0917,3.3452,True,True
2,2,-0.0647,0.0999,-0.6479,-0.1753,0.0917,-1.9118,False,False
3,3,-0.0717,0.1003,-0.7147,0.0062,0.0917,0.0672,False,False
4,4,0.1046,0.1007,1.0384,0.1333,0.0917,1.4546,False,False
5,5,0.0841,0.1016,0.8280,-0.0095,0.0917,-0.1039,False,False
6,6,0.0228,0.1022,0.2235,0.0199,0.0917,0.2172,False,False
7,7,-0.1326,0.1022,-1.2971,-0.1397,0.0917,-1.5234,False,False


## Practice Prompt

A stationary working series has ACF spikes at lags 1 and 2 and then no clear spikes, while PACF dies down gradually. Which tentative model is suggested?

Reference idea: this is the classical MA(2) pattern, because the ACF cuts off after lag 2 and the PACF dies down.